# 04 - Explore the average service offer

**Source:** `offre_hebdomadaire_moyenne_hors_vacances.csv`.
**Default scope: first 20,000 rows only.**
**Module:** `scripts/datasets/service_offer.py`.

Inspect stop/line IDs, hours, coordinates and daily average-service columns. A first-row
sample has incomplete geographic and hourly coverage. Understand the source grain before
combining repeated rows or interpreting service frequency.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import plotly.io as pio
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Open the hackathon project folder before running this notebook.')
if not Path(sys.executable).resolve().is_relative_to(ROOT / '.venv'):
    raise RuntimeError('Select the project .venv kernel, then restart and run all.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.datasets import service_offer
from scripts.exploration import column_summary, repeated_keys, save_parquet

pio.renderers.default = 'plotly_mimetype'
print('Python:', sys.executable)


Python: d:\Hackathon Mobilités 2026 Spécial Dataviz\.venv\Scripts\python.exe


In [2]:
source = ROOT / service_offer.FILENAME
SAMPLE_ROWS = 20_000
raw = service_offer.load_sample(source, rows=SAMPLE_ROWS)
print('SCOPE: first', SAMPLE_ROWS, 'rows; geographic and hourly coverage may be incomplete.')


SCOPE: first 20000 rows; geographic and hourly coverage may be incomplete.


In [3]:
print('Loaded rows and columns:', raw.shape)
display(raw.head())
raw.info()


Loaded rows and columns: (20000, 22)


,code_mode_ligne,libelle_mode_ligne,id_ligne_commerciale,nom_ligne_commerciale,id_zone_de_correspondance,id_zone_arret,id_arret,nom_arret,tranche_horaire,nb_courses_lundi,...,nb_courses_jeudi,nb_courses_vendredi,nb_courses_samedi,nb_courses_dimanche,code_departement,code_commune,nom_commune,longitude_arret,latitude_arret,epci
0,3,Bus,C00001,482,463511.0,463510,463509,Cimetières,8,4.0,...,4.0,4.0,1.0,1.0,94,94077,Villeneuve-le-Roi,2.4085233541297746,48.74187509162765,Métropole du Grand Paris
1,3,Bus,C00001,482,463511.0,463510,463509,Cimetières,10,2.0,...,2.0,2.0,2.0,1.0,94,94077,Villeneuve-le-Roi,2.4085233541297746,48.74187509162765,Métropole du Grand Paris
2,3,Bus,C00001,482,463511.0,463510,463509,Cimetières,22,1.0,...,1.0,1.0,1.0,0.0,94,94077,Villeneuve-le-Roi,2.4085233541297746,48.74187509162765,Métropole du Grand Paris
3,3,Bus,C00001,482,463511.0,463510,463512,Cimetières,10,2.0,...,2.0,2.0,2.0,1.0,94,94077,Villeneuve-le-Roi,2.408657302643461,48.74205564258887,Métropole du Grand Paris
4,3,Bus,C00001,482,463511.0,463510,463512,Cimetières,20,2.0,...,2.0,2.0,1.0,1.0,94,94077,Villeneuve-le-Roi,2.408657302643461,48.74205564258887,Métropole du Grand Paris


<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   code_mode_ligne            20000 non-null  string
 1   libelle_mode_ligne         20000 non-null  string
 2   id_ligne_commerciale       20000 non-null  string
 3   nom_ligne_commerciale      20000 non-null  string
 4   id_zone_de_correspondance  20000 non-null  string
 5   id_zone_arret              20000 non-null  string
 6   id_arret                   20000 non-null  string
 7   nom_arret                  20000 non-null  string
 8   tranche_horaire            20000 non-null  string
 9   nb_courses_lundi           20000 non-null  string
 10  nb_courses_mardi           20000 non-null  string
 11  nb_courses_mercredi        20000 non-null  string
 12  nb_courses_jeudi           20000 non-null  string
 13  nb_courses_vendredi        20000 non-null  string
 14  nb_courses_samedi

## Clean a copy and inspect the columns

The original file and `raw` table stay unchanged. Strip surrounding spaces and turn
blank cells into missing values; preserve special codes such as `NA` and `ZZ`.
Dataset-specific numeric conversions are documented in the module's `clean()` function.
Unexpected numeric text raises an error for inspection. No rows are dropped or deduplicated.


In [4]:
cleaned = service_offer.clean(raw)
assert len(cleaned) == len(raw)
display(cleaned.head())
display(column_summary(cleaned))


,code_mode_ligne,libelle_mode_ligne,id_ligne_commerciale,nom_ligne_commerciale,id_zone_de_correspondance,id_zone_arret,id_arret,nom_arret,tranche_horaire,nb_courses_lundi,...,nb_courses_samedi,nb_courses_dimanche,code_departement,code_commune,nom_commune,longitude_arret,latitude_arret,epci,coordinates_in_bounds,negative_offer_value
0,3,Bus,C00001,482,463511.0,463510,463509,Cimetières,8,4.0,...,1.0,1.0,94,94077,Villeneuve-le-Roi,2.408523,48.741875,Métropole du Grand Paris,True,False
1,3,Bus,C00001,482,463511.0,463510,463509,Cimetières,10,2.0,...,2.0,1.0,94,94077,Villeneuve-le-Roi,2.408523,48.741875,Métropole du Grand Paris,True,False
2,3,Bus,C00001,482,463511.0,463510,463509,Cimetières,22,1.0,...,1.0,0.0,94,94077,Villeneuve-le-Roi,2.408523,48.741875,Métropole du Grand Paris,True,False
3,3,Bus,C00001,482,463511.0,463510,463512,Cimetières,10,2.0,...,2.0,1.0,94,94077,Villeneuve-le-Roi,2.408657,48.742056,Métropole du Grand Paris,True,False
4,3,Bus,C00001,482,463511.0,463510,463512,Cimetières,20,2.0,...,1.0,1.0,94,94077,Villeneuve-le-Roi,2.408657,48.742056,Métropole du Grand Paris,True,False


,data_type,missing_rows,distinct_values
column,,,
code_mode_ligne,string,0,1
libelle_mode_ligne,string,0,1
id_ligne_commerciale,string,0,37
nom_ligne_commerciale,string,0,73
id_zone_de_correspondance,string,0,481
id_zone_arret,string,0,506
id_arret,string,0,906
nom_arret,string,0,495
tranche_horaire,Int64,0,23


## Inspect measurements and row grain

Missing counts remain missing, not zero. Decimal average counts can be legitimate.
Identifiers stay as supplied, including decimal-looking text. Hours above 23 remain
unchanged until the service-day convention is documented. Repeated keys need investigation.


In [5]:
display(cleaned[service_offer.COURSE_COLUMNS].describe())
display(cleaned['libelle_mode_ligne'].value_counts(dropna=False).rename('sample_records'))
display(repeated_keys(cleaned, ['id_ligne_commerciale', 'id_arret', 'tranche_horaire']).head(20))
display(cleaned.loc[~cleaned.coordinates_in_bounds | cleaned.negative_offer_value].head(20))


,nb_courses_lundi,nb_courses_mardi,nb_courses_mercredi,nb_courses_jeudi,nb_courses_vendredi,nb_courses_samedi,nb_courses_dimanche
count,20000.0,20000.0,20000.0,20000.0,20000.0,20000.0,20000.0
mean,1.751619,1.75194,1.744769,1.75047,1.759423,0.753402,0.40862
std,1.786856,1.788887,1.78675,1.787301,1.789498,1.17402,0.801049
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,0.285714,0.285714,0.285714,0.30303,0.266667,0.0,0.0
50%,1.0,1.0,1.0,1.0,1.0,0.0,0.0
75%,3.0,3.0,3.0,3.0,3.0,1.0,0.6
max,17.028571,17.028571,17.028571,17.030303,17.033333,10.0,6.966667


libelle_mode_ligne
Bus    20000
Name: sample_records, dtype: int64[pyarrow]

,id_ligne_commerciale,id_arret,tranche_horaire,records
13163,C00046,42010,20,4
13181,C00046,470599,21,4
13180,C00046,470599,20,4
13179,C00046,470599,19,4
13162,C00046,42010,19,4
13161,C00046,42010,18,4
13160,C00046,42010,17,4
12810,C00046,12712,5,4
12811,C00046,12712,6,4
12812,C00046,12712,7,4


,code_mode_ligne,libelle_mode_ligne,id_ligne_commerciale,nom_ligne_commerciale,id_zone_de_correspondance,id_zone_arret,id_arret,nom_arret,tranche_horaire,nb_courses_lundi,...,nb_courses_samedi,nb_courses_dimanche,code_departement,code_commune,nom_commune,longitude_arret,latitude_arret,epci,coordinates_in_bounds,negative_offer_value


In [6]:
hour_summary = service_offer.hour_coverage(cleaned)
display(hour_summary)
service_offer.plot_hour_coverage(hour_summary, scope=f'first {len(cleaned):,} records').show()


,tranche_horaire,source_records
0,4,38
1,5,531
2,6,1091
3,7,1522
4,8,1409
5,9,1165
6,10,925
7,11,973
8,12,1330
9,13,1319


## Play with a line and stop in the sample

This shows raw rows, not a timetable. Do not sum counts over successive stops to estimate
departures or passengers. Average counts cannot establish exact transfers. The hour-summary
table gives the values behind the chart, which shows source-row coverage, not frequency.


In [7]:
LINE_ID = cleaned['id_ligne_commerciale'].dropna().iloc[0]
line_rows = cleaned.loc[cleaned.id_ligne_commerciale.eq(LINE_ID)]
STOP_ID = line_rows['id_arret'].dropna().iloc[0]
selected = line_rows.loc[line_rows.id_arret.eq(STOP_ID)].sort_values('tranche_horaire')
display(selected[['id_ligne_commerciale', 'id_arret', 'nom_arret', 'tranche_horaire',
                  *service_offer.COURSE_COLUMNS]])


,id_ligne_commerciale,id_arret,nom_arret,tranche_horaire,nb_courses_lundi,nb_courses_mardi,nb_courses_mercredi,nb_courses_jeudi,nb_courses_vendredi,nb_courses_samedi,nb_courses_dimanche
176,C00001,463509,Cimetières,5,4.0,4.0,4.0,4.0,4.0,1.0,0.0
711,C00001,463509,Cimetières,6,4.0,4.0,4.0,4.0,4.0,1.0,0.0
712,C00001,463509,Cimetières,7,4.0,4.0,4.0,4.0,4.0,1.0,0.966667
0,C00001,463509,Cimetières,8,4.0,4.0,4.0,4.0,4.0,1.0,1.0
177,C00001,463509,Cimetières,9,3.0,3.0,3.0,3.0,3.0,2.0,1.0
1,C00001,463509,Cimetières,10,2.0,2.0,2.0,2.0,2.0,2.0,1.0
713,C00001,463509,Cimetières,11,2.0,2.0,2.0,2.0,2.0,2.0,1.0
714,C00001,463509,Cimetières,12,2.0,2.0,2.0,2.0,2.0,2.0,1.0
527,C00001,463509,Cimetières,13,2.0,2.0,2.0,2.0,2.0,2.0,1.0
178,C00001,463509,Cimetières,14,2.0,2.0,2.0,2.0,2.0,2.0,1.0


## Optional: count full-source hour coverage in chunks

This counts source records and still uses only this source. It does not sum offer values.


In [8]:
RUN_FULL_SCAN = False
if RUN_FULL_SCAN:
    parts = []
    for chunk in service_offer.read_chunks(source):
        parts.append(service_offer.hour_coverage(service_offer.clean(chunk)))
    full_hour_summary = pd.concat(parts).groupby('tranche_horaire', dropna=False)[
        'source_records'
    ].sum().reset_index()
    display(full_hour_summary)
else:
    print('Full scan is off. Chart and tables above describe the sample only.')


Full scan is off. Chart and tables above describe the sample only.


## Optional: save this dataset as Parquet



In [9]:
SAVE_CLEANED = False
output_path = ROOT / 'data' / 'cleaned' / 'service_offer_sample.parquet'

if SAVE_CLEANED:
    save_parquet(cleaned, output_path)
    print('Saved', len(cleaned), 'rows to', output_path)
else:
    print('No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.')


No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.
